# Download Yelp Open Dataset → Databricks Volume

**Project:** Using Yelp Data to Estimate Business Demand and Identify Key Drivers
**Course:** DAMO-699-1 Capstone (Spring 2026)
**Team:** Sunghoon Ahn, Eun Soo Park, Thiri Htet Oo, Jayanthi Muthuselvan

This notebook downloads the **Yelp Open Dataset (April 2024)** as raw JSON files from the HuggingFace mirror (`adamamer20/yelp-04-2024`) and saves them to the Databricks Volume `/Volumes/workspace/default/yelp/`. Subsequent cleaning, EDA, and modeling notebooks read from this location.

**Why HuggingFace mirror?** Yelp's official dataset page requires a manual form submission per download. The HuggingFace mirror serves the same April 2024 release and is scriptable, which makes it suitable for a reproducible team workflow.

**Expected wall time:** ~30–60 minutes depending on cluster bandwidth (5 files, ~9.2 GB total). The largest file (`review.json` ≈ 5.3 GB) dominates the run time. Each file shows its own progress bar via `huggingface_hub`.


---
## 1. Setup

Install `huggingface_hub` and restart Python so the new package is available in the notebook session. Imports and paths are then defined.


In [0]:
%pip install huggingface_hub --quiet


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 2.11.4 requires protobuf<5,>=3.12.0, but you have protobuf 5.29.3 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
from huggingface_hub import hf_hub_download
import os
import shutil
import time

# HuggingFace dataset repo (Yelp Open Dataset, April 2024)
REPO_ID = "adamamer20/yelp-04-2024"

# Target Volume — all downloaded JSON files land directly here (flat layout)
TARGET_DIR = "/Volumes/workspace/default/yelp"

# Files to download. Ordered smallest → largest so quick wins come first
# and any auth or path issue surfaces before the multi-GB downloads start.
FILES = [
    ("business", "yelp_dataset/yelp_academic_dataset_business.json", "113 MB"),
    ("tip",      "yelp_dataset/yelp_academic_dataset_tip.json",      "172 MB"),
    ("checkin",  "yelp_dataset/yelp_academic_dataset_checkin.json",  "274 MB"),
    ("user",     "yelp_dataset/yelp_academic_dataset_user.json",     "3.4 GB"),
    ("review",   "yelp_dataset/yelp_academic_dataset_review.json",   "5.3 GB"),
]

print(f"Target Volume:  {TARGET_DIR}")
print(f"Source repo:    huggingface.co/datasets/{REPO_ID}")
print(f"Files to fetch: {len(FILES)}")


Target Volume:  /Volumes/workspace/default/yelp
Source repo:    huggingface.co/datasets/adamamer20/yelp-04-2024
Files to fetch: 5


---
## 2. Create Target Volume (if missing)

The Volume must exist before the JSON files can be written. `CREATE VOLUME IF NOT EXISTS` is idempotent — safe to run repeatedly. If the workspace/schema differs from `workspace.default`, adjust the SQL accordingly.


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.yelp")

os.makedirs(TARGET_DIR, exist_ok=True)
print(f"Volume ready: {TARGET_DIR}")


Volume ready: /Volumes/workspace/default/yelp


---
## 3. Download JSON Files

Each file is fetched with `hf_hub_download`, which streams the file from HuggingFace and shows its own progress bar. The HuggingFace SDK preserves the source repo path, so the file initially lands at `{TARGET_DIR}/yelp_dataset/...`; the cell then moves it to the flat target location `{TARGET_DIR}/...` so all five JSON files sit side by side.

Re-running the cell skips files that already exist, so an interrupted run can resume without re-downloading.


In [0]:
total_start = time.time()

for i, (name, repo_path, size_label) in enumerate(FILES, 1):
    filename   = os.path.basename(repo_path)
    final_path = f"{TARGET_DIR}/{filename}"

    # Skip if already downloaded (idempotent re-runs)
    if os.path.exists(final_path):
        size_mb = os.path.getsize(final_path) / (1024 ** 2)
        print(f"[{i}/{len(FILES)}] {name:<8s}  SKIP  ({size_mb:>7,.1f} MB)")
        continue

    # Fetch from HuggingFace (tqdm progress bar prints during download)
    start = time.time()
    hf_hub_download(
        repo_id=REPO_ID,
        filename=repo_path,
        repo_type="dataset",
        local_dir=TARGET_DIR,
    )

    # Flatten: move from {TARGET_DIR}/yelp_dataset/file.json → {TARGET_DIR}/file.json
    nested_path = f"{TARGET_DIR}/yelp_dataset/{filename}"
    if os.path.exists(nested_path):
        shutil.move(nested_path, final_path)

    elapsed = time.time() - start
    size_mb = os.path.getsize(final_path) / (1024 ** 2)
    print(f"[{i}/{len(FILES)}] {name:<8s}  DONE  ({size_mb:>7,.1f} MB, {elapsed/60:.1f} min)")

# Remove the now-empty yelp_dataset/ subdirectory if it exists
nested_dir = f"{TARGET_DIR}/yelp_dataset"
if os.path.isdir(nested_dir) and not os.listdir(nested_dir):
    os.rmdir(nested_dir)

print(f"\nTotal: {(time.time() - total_start) / 60:.1f} min")


yelp_dataset/yelp_academic_dataset_busin(…):   0%|          | 0.00/119M [00:00<?, ?B/s]

[1/5] business  DONE  (  113.4 MB, 0.1 min)


yelp_dataset/yelp_academic_dataset_tip.j(…):   0%|          | 0.00/181M [00:00<?, ?B/s]

[2/5] tip       DONE  (  172.2 MB, 0.1 min)


yelp_dataset/yelp_academic_dataset_check(…):   0%|          | 0.00/287M [00:00<?, ?B/s]

[3/5] checkin   DONE  (  273.7 MB, 0.2 min)


yelp_dataset/yelp_academic_dataset_user.(…):   0%|          | 0.00/3.36G [00:00<?, ?B/s]

[4/5] user      DONE  (3,207.5 MB, 0.2 min)


yelp_dataset/yelp_academic_dataset_revie(…):   0%|          | 0.00/5.34G [00:00<?, ?B/s]

[5/5] review    DONE  (5,094.4 MB, 0.3 min)

Total: 1.1 min


---
## 4. Verify Downloaded Files

List the final contents of the Volume to confirm all five JSON files landed at the expected flat path and their sizes match the documented Yelp Open Dataset (April 2024) release.


In [0]:
print(f"Contents of {TARGET_DIR}:")
print("-" * 70)

total_size = 0
files_found = 0

for f in sorted(os.listdir(TARGET_DIR)):
    full_path = f"{TARGET_DIR}/{f}"
    if os.path.isfile(full_path):
        size_mb = os.path.getsize(full_path) / (1024 ** 2)
        total_size += size_mb
        files_found += 1
        print(f"  {f:<50s} {size_mb:>10,.1f} MB")

print("-" * 70)
print(f"  {'TOTAL':<50s} {total_size:>10,.1f} MB  ({total_size/1024:.2f} GB)")
print(f"  Files: {files_found}")


Contents of /Volumes/workspace/default/yelp:
----------------------------------------------------------------------
  yelp_academic_dataset_business.json                     113.4 MB
  yelp_academic_dataset_checkin.json                      273.7 MB
  yelp_academic_dataset_review.json                     5,094.4 MB
  yelp_academic_dataset_tip.json                          172.2 MB
  yelp_academic_dataset_user.json                       3,207.5 MB
----------------------------------------------------------------------
  TOTAL                                                 8,861.2 MB  (8.65 GB)
  Files: 5
